# Week 7 — 時間序列診斷

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 比較價格序列與報酬序列的統計行為。
- 計算並解讀自相關函數 (ACF)。
- 計算 rolling 波動度並觀察波動度叢聚。
- 比較定態（AR(1)）與非定態（隨機漫步）序列。

## 預估學習時間

約 8–10 小時。

## 先備概念

- Week 1 的報酬
- Week 3 的隨機變數

## 外部學習資源

- [Forecasting: Principles and Practice, the Pythonic Way](https://otexts.com/fpppy/)
- [Penn State STAT 510 Applied Time Series Analysis](https://online.stat.psu.edu/stat510/)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

In [ ]:
# 教學樣式設定（CJK 字型、負號正常顯示、固定隨機種子）
import matplotlib as _mpl
_mpl.rcParams['font.sans-serif'] = [
    'PingFang TC', 'Heiti TC', 'Microsoft JhengHei',
    'Noto Sans CJK TC', 'Noto Sans TC',
    'WenQuanYi Zen Hei', 'Source Han Sans TC',
    'Arial Unicode MS', 'DejaVu Sans',
]
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## 概念說明

### 定態性 (stationarity)

**定態**序列的統計性質（平均、變異數、自相關）不隨時間改變。價格序列通常**非定態**（有趨勢、會漂移）；報酬序列通常**較接近定態**。

### 自相關函數 (ACF)

ACF 衡量序列與自身落後值的相關性：

$$ \rho_k = \frac{\operatorname{Cov}(x_t, x_{t-k})}{\operatorname{Var}(x_t)}. $$

**白噪音**在所有非零落後的 ACF 都接近 0。

### 為什麼隨機切分不適用

時間序列有順序。隨機 train/test 切分會讓模型「看到未來」，造成 look-ahead bias——這是 Week 8 的核心主題。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.data import (
    SyntheticConfig, generate_correlated_prices,
    generate_ar1_series, generate_random_walk,
)
from quant_math_roadmap.finance.returns import simple_returns
from quant_math_roadmap.time_series.diagnostics import (
    adf_stationarity_test, autocorrelation_function,
    rolling_volatility,
)

config = SyntheticConfig(n_assets=1, n_periods=756, seed=21,
                         vol_regime_multiplier=2.0)
prices = generate_correlated_prices(config).iloc[:, 0]
returns = simple_returns(prices)

### 價格 vs 報酬序列

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(prices.index, prices.values)
axes[0].set_title('價格序列（通常非定態：有趨勢）')
axes[0].set_ylabel('價格')
axes[1].plot(returns.index, returns.values)
axes[1].set_title('報酬序列（較接近定態，但有波動度叢聚）')
axes[1].set_xlabel('日期')
axes[1].set_ylabel('每日報酬')
plt.tight_layout()
plt.show()

### ADF 定態性檢定

In [ ]:
price_adf = adf_stationarity_test(prices)
return_adf = adf_stationarity_test(returns)
print('價格序列 ADF p-value :', round(price_adf['p_value'], 4))
print('報酬序列 ADF p-value :', round(return_adf['p_value'], 4))
print('小 p-value = 有證據反對單根（傾向定態）。')

### 自相關函數

In [ ]:
acf_returns = autocorrelation_function(returns, max_lag=20)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(acf_returns.index, acf_returns.values)
ax.set_title('報酬序列的自相關函數 (ACF)')
ax.set_xlabel('落後期數 lag')
ax.set_ylabel('自相關')
plt.show()
print('報酬的 ACF 在非零 lag 多半接近 0 — 接近白噪音。')

### Rolling 波動度與波動度叢聚

In [ ]:
roll_vol = rolling_volatility(returns, window=40)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(roll_vol.index, roll_vol.values, label='40 期 rolling 波動度')
ax.set_title('Rolling 波動度：波動度叢聚')
ax.set_xlabel('日期')
ax.set_ylabel('rolling 標準差')
ax.legend()
plt.show()
print('合成資料在後半段加入了波動度 regime shift，這裡清楚可見。')

### 定態 vs 非定態：AR(1) vs 隨機漫步

In [ ]:
ar1 = generate_ar1_series(600, phi=0.6, seed=5)
walk = generate_random_walk(600, drift=0.0, seed=5)

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(ar1.index, ar1.values)
axes[0].set_title('AR(1), phi=0.6（定態：會回到平均）')
axes[0].set_ylabel('數值')
axes[1].plot(walk.index, walk.values)
axes[1].set_title('隨機漫步（非定態：會漂移、不回頭）')
axes[1].set_xlabel('日期')
axes[1].set_ylabel('數值')
plt.tight_layout()
plt.show()
print('AR(1)  ADF p-value:', round(adf_stationarity_test(ar1)['p_value'], 4))
print('隨機漫步 ADF p-value:', round(adf_stationarity_test(walk)['p_value'], 4))

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 用自己的話定義定態性，並說明為何價格通常非定態。
2. 白噪音的 ACF 長什麼樣子？
3. 解釋什麼是波動度叢聚。

### 應用練習

In [ ]:
# 應用練習 1：產生 phi=0.0、phi=0.5、phi=0.9 三種 AR(1)，
# 計算各自 lag-1 的自相關，確認它接近 phi。
from quant_math_roadmap.time_series.diagnostics import autocorrelation
for phi in [0.0, 0.5, 0.9]:
    series = generate_ar1_series(4000, phi=phi, seed=1)
    ac1 = None  # TODO: autocorrelation(series, 1)
    print(f'phi={phi}: lag-1 autocorr = {ac1}')

In [ ]:
# 應用練習 2：對價格序列與報酬序列各畫 ACF，比較兩者的差異。
acf_price = None  # TODO: autocorrelation_function(prices, max_lag=20)
if acf_price is not None:
    print('價格 lag-1 自相關:', round(acf_price.iloc[1], 4))
    print('報酬 lag-1 自相關:', round(acf_returns.iloc[1], 4))

### 反思問題

1. 既然報酬序列的 ACF 幾乎都接近 0，這對「用過去報酬預測未來報酬」的策略有什麼啟示？

## 小測驗（自我檢核）
回答下面的選擇題，然後執行下一格自動對答案。答案以雜湊儲存，不會直接洩漏。

**Q1. 定態（stationary）序列的特徵是？**
- A. 價格永遠上漲
- B. 統計性質（平均、變異數、自相關）不隨時間改變
- C. 完全沒有波動
- D. 沒有任何自相關

**Q2. 白噪音的 ACF 在非零 lag 應該？**
- A. 接近 0
- B. 接近 1
- C. 隨 lag 遞增
- D. 全部為負

**Q3. AR(1)：x_t = φx_{t−1} + ε_t 平穩的條件是？**
- A. φ > 0
- B. |φ| < 1
- C. φ = 1
- D. φ > 1

**Q4. 「波動度叢聚」指的是？**
- A. 報酬集中在平均值附近
- B. 高波動期與低波動期各自成群出現
- C. 價格聚集在整數關卡
- D. 自相關為零

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: 填入 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: '8652b328d7d88f86', 2: '917be34eb394105b', 3: 'cb788c82b06bc75e', 4: '65bc6a6a84b4ae5a'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: 未作答')
        continue
    _h = _hashlib.sha256(f'qmr-w7-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ 正確' if _ok else '✘ 不正確'))
print(f'得分: {_n_correct} / {len(my_answers)}')

## 常見錯誤

- **直接對非定態的價格序列建模，而不先轉成報酬。**
- **對時間序列用隨機 train/test 切分。**
- **rolling 計算時把前面視窗不足的 NaN 用未來值回填。**
- **把報酬微弱的自相關過度解讀成可獲利訊號。**

## 完成本週後，你應該能做到什麼

- [ ] 能解釋定態性並判斷價格 vs 報酬。
- [ ] 能計算並解讀 ACF。
- [ ] 能計算 rolling 波動度並辨識波動度叢聚。
- [ ] 能說明隨機切分為何不適用於時間序列。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../docs/math/) 與 [`docs/finance/`](../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。